## Modelo predictivo de coste de seguro

## Cargamos el CSV y las librerias

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_validate, learning_curve
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from scipy.stats import loguniform
from sklearn.linear_model import Ridge
from sklearn.linear_model import RidgeCV

In [2]:
df= pd.read_csv("Data/df_con_probabilidad.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101886 entries, 0 to 101885
Data columns (total 24 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   ID                                     101886 non-null  str    
 1   Edad                                   101886 non-null  int64  
 2   Ingresos                               101886 non-null  float64
 3   Monto_Inicial                          101886 non-null  int64  
 4   Scoring_Crediticio                     101886 non-null  int64  
 5   Meses_Empleo                           101886 non-null  float64
 6   Num_Creditos                           101886 non-null  int64  
 7   Ratio_Interes                          101886 non-null  float64
 8   Duracion                               101886 non-null  float64
 9   Ratio_Deuda_Ingresos                   101886 non-null  float64
 10  Estudios                               101886 non-null  int64  
 11

In [4]:
df= df.drop(columns="Impago")

Eliminamos la columna impago porque es una clasificacion poco realista, Esto es debido a que la clasificacion depende de un umbral que nosotros mismos hemos elegido a proposito para que trate de detectar el maximo numero de impagos posibles. Es por eso que vamos a trabajar unicamente con la probabilidad de impago que nuestro modelo nos ha devuelto

Aun asi viendo las variables me he dado cuenta que a la hora de hacer un get dummies de las variables categoricas se nos olvido eliminar una columna por lo que vamos a eliminar una de cada una. Esto es necesario porque de no hacerlo la suma de las categorias siempre sera 1 porque imaginate si tengo tres categorias (en formato dummy) y se dos de ellas no necesito la tercera 

Para ello he decidido eliminar la mas frecuente en ambas para que sea una referencia intuitiva

In [5]:
cols_estado_civil = ["Estado_Civil_Casado", "Estado_Civil_Divorciado", "Estado_Civil_Soltero"]
cols_jornada = ["Tipo_Jornada_Laboral_Autónomo", "Tipo_Jornada_Laboral_Desempleado",
                "Tipo_Jornada_Laboral_Jornada completa", "Tipo_Jornada_Laboral_Tiempo parcial"]

In [6]:
estado_civil_media = df[cols_estado_civil].mean()
print(estado_civil_media)

Estado_Civil_Casado        0.511778
Estado_Civil_Divorciado    0.186424
Estado_Civil_Soltero       0.301798
dtype: float64


In [7]:
df= df.drop(columns="Estado_Civil_Casado")

In [8]:
jornada_media = df[cols_jornada].mean()
print(jornada_media)

Tipo_Jornada_Laboral_Autónomo            0.148519
Tipo_Jornada_Laboral_Desempleado         0.099513
Tipo_Jornada_Laboral_Jornada completa    0.550056
Tipo_Jornada_Laboral_Tiempo parcial      0.201912
dtype: float64


In [9]:
df= df.drop(columns="Tipo_Jornada_Laboral_Jornada completa")

## Separamos en TRAIN TEST y PUBLICO

In [10]:
df_modelo = df.copy()

df_modelo = df_modelo.drop(columns="ID")


df_publico = df_modelo.sample(frac=0.10, random_state=42)
df_modelo = df_modelo.drop(df_publico.index)


target = "Prima"
X = df_modelo.drop(columns=target)
y = df_modelo[target]

X_publico = df_publico.drop(columns=target)
y_publico = df_publico[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Regresion simple

In [11]:
columnas_a_escalar = ["Edad", "Ingresos", "Monto_Inicial", "Scoring_Crediticio",
                      "Num_Creditos", "Ratio_Interes", "Duracion", "Ratio_Deuda_Ingresos"]

scaler = StandardScaler()

X_train_scal = X_train.copy()
X_test_scal = X_test.copy()
X_publico_scal = X_publico.copy()

X_train_scal[columnas_a_escalar] = scaler.fit_transform(X_train[columnas_a_escalar])
X_test_scal[columnas_a_escalar] = scaler.transform(X_test[columnas_a_escalar])
X_publico_scal[columnas_a_escalar] = scaler.transform(X_publico[columnas_a_escalar])

In [12]:
X_train_scaled = sm.add_constant(X_train_scal, prepend=True)
modelo_lin_reg = sm.OLS(endog=y_train, exog=X_train_scaled)
modelo_lin_reg = modelo_lin_reg.fit()
print(modelo_lin_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  Prima   R-squared:                       0.677
Model:                            OLS   Adj. R-squared:                  0.677
Method:                 Least Squares   F-statistic:                     8102.
Date:                Mon, 02 Nov 2026   Prob (F-statistic):               0.00
Time:                        08:35:05   Log-Likelihood:            -4.7248e+05
No. Observations:               73357   AIC:                         9.450e+05
Df Residuals:                   73337   BIC:                         9.452e+05
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
co

Vamos a quitar las variables con un p valor mas alto que 0,05 ya que si este es mas alto que 0,05 no podriamos rechazar la hipotesis nula y por lo tanto esas variables no serian estadisticamente significativas

In [13]:
variables_quitar = [
    "Meses_Empleo",
    "Num_Creditos",
    "Posesion_Hipoteca",
    "Personas_Cargo",
    "Fiador",
    "Estado_Civil_Divorciado",
    "Estado_Civil_Soltero",
    "Tipo_Jornada_Laboral_Autónomo",
    "Tipo_Jornada_Laboral_Desempleado",
    "Tipo_Jornada_Laboral_Tiempo parcial"
    
]

In [14]:
X_train_hipotesisnula =X_train_scal.drop(columns=variables_quitar)
X_test_hipotesisnula = X_test_scal.drop(columns=variables_quitar)

In [15]:
X_train_scaled = sm.add_constant(X_train_hipotesisnula, prepend=True)
modelo_lin_reg = sm.OLS(endog=y_train, exog=X_train_scaled)
modelo_lin_reg = modelo_lin_reg.fit()
print(modelo_lin_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  Prima   R-squared:                       0.677
Model:                            OLS   Adj. R-squared:                  0.677
Method:                 Least Squares   F-statistic:                 1.711e+04
Date:                Mon, 02 Nov 2026   Prob (F-statistic):               0.00
Time:                        08:35:05   Log-Likelihood:            -4.7248e+05
No. Observations:               73357   AIC:                         9.450e+05
Df Residuals:                   73347   BIC:                         9.451e+05
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                  424.6818 

## Regresion Polinomica

In [19]:
def comparar_grados(X, y, grados=(3,4, 5)):

    
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

   
    resultados = []

    
    for g in grados:
        
        poly = PolynomialFeatures(degree=g, include_bias=False)
        X_poly = poly.fit_transform(X)

       
        scores = cross_validate(LinearRegression(), X_poly, y, cv=cv, return_train_score=True,
                                scoring={"rmse": "neg_root_mean_squared_error",
                                         "mae": "neg_mean_absolute_error",
                                         "r2": "r2"})

   
        resultados.append({
            "grado": g,
            "n_variables": X_poly.shape[1],               
            "RMSE_train": -scores["train_rmse"].mean(),   
            "RMSE_cv": -scores["test_rmse"].mean(),       
            "MAE_cv": -scores["test_mae"].mean(),         
            "R2_cv": scores["test_r2"].mean()             
        })

   
    return pd.DataFrame(resultados)

resultados = comparar_grados(X_train_hipotesisnula, y_train)
resultados

,grado,n_variables,RMSE_train,RMSE_cv,MAE_cv,R2_cv
0,3,219,74.580196,74.996857,55.822367,0.921121
1,4,714,63.302325,64.363331,47.005303,0.941903
2,5,2001,56.861266,59.928520,42.573170,0.949635


In [20]:
mejor_grado = int(resultados.loc[resultados["RMSE_cv"].idxmin(), "grado"])

# Generamos las variables polinómicas con el grado elegido
poly = PolynomialFeatures(degree=mejor_grado, include_bias=False)
X_train_poly = poly.fit_transform(X_train_hipotesisnula)  
X_test_poly = poly.transform(X_test_hipotesisnula)        

# Entrenamos el modelo final con todo el conjunto de entrenamiento
modelo_final = LinearRegression()
modelo_final.fit(X_train_poly, y_train)

# Predecimos sobre el test: datos que el modelo no ha visto en ningún momento
pred_test = modelo_final.predict(X_test_poly)

# Evaluamos el modelo una única vez en test: esta es la estimación final de su rendimiento con datos nuevos
print(f"Mejor grado: {mejor_grado}")
print(f"Test:  RMSE = {root_mean_squared_error(y_test, pred_test):.2f} | "   
      f"MAE = {mean_absolute_error(y_test, pred_test):.2f} | "               
      f"R² = {r2_score(y_test, pred_test):.3f}")    

Mejor grado: 5
Test:  RMSE = 58.46 | MAE = 41.86 | R² = 0.952


## Regularización 

In [21]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)


In [22]:
modelo_ridgecv = RidgeCV(
            alphas           = np.logspace(-20, 2, 200),
            scoring          ='neg_root_mean_squared_error',
            fit_intercept    = True,
            store_cv_results = True
         )

modelo_ridgecv.fit(X = X_train_hipotesisnula, y = y_train)

,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.For an example on how regularization strength affects the model coefficients,see :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`.",array([1.0000...00000000e+02])
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error <mean_squared_error>` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination <r2_score>` (:math:`R^2`) otherwise.",'neg_root_mean_squared_error'
,"store_cv_results store_cv_results: bool, default=FalseFlag indicating if the cross-validation values corresponding toeach alpha should be stored in the ``cv_results_`` attribute (seebelow). This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionchanged:: 1.5 Parameter name changed from `store_cv_values` to `store_cv_results`.",True
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used, else,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.",None
,"gcv_mode gcv_mode: {'auto', 'svd', 'eigen'}, default='auto'Flag indicating which strategy to use when performingLeave-One-Out Cross-Validation. Options are:: 'auto' : same as 'eigen' 'svd' : use singular value decomposition of X when X is dense, fallback to 'eigen' when X is sparse 'eigen' : use eigendecomposition of X X' when n_samples <= n_features or X' X when n_features < n_samplesThe 'auto' mode is the default and is intended to pick the cheaperoption depending on the shape and sparsity of the training data.",None
,"alpha_per_target alpha_per_target: bool, default=FalseFlag indicating whether to optimize the alpha value (picked from the`alphas` parameter list) for each target separately (for multi-outputsettings: multiple prediction targets). When set to `True`, afterfitting, the `alpha_` attribute will contain a value for each target.When set to `False`, a single alpha is used for all targets.This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionadded:: 0.24",False
Name,Type,Value
"alpha_ alpha_: float or ndarray of shape (n_targets,)Estimated regularization parameter, or, if ``alpha_per_target=True``,the estimated regularization parameter for each target.",float,0.1035
"best_score_ best_score_: float or ndarray of shape (n_targets,)Score of base estimator with best alpha, or, if``alpha_per_target=True``, a score for each target... versionadded:: 0.23",float,-151.7
"coef_ coef_: ndarray of shape (n_features) or (n_targets, n_features)Weight vector(s).","ndarray[float64](9,)","[ -12.17, 18.02, 93

In [23]:
modelo_ridgecv.alpha_

0.10353218432956615